# 04 — Fine-tune from a Published Checkpoint

Published cage-fusion weights capture a rich chemical representation.
Instead of training from scratch, load the encoder from a published model
and adapt it to your task — often converging faster and to a better optimum.

**What you'll learn**
- `load_backbone_only=True` — load encoder weights, reset the task head
- `staged_finetune` — the canonical 4-phase freeze/unfreeze protocol
- Manual freeze/unfreeze with `freeze_phase` + `rebuild_optimizer`
- Compare fine-tuned vs. from-scratch on a held-out set

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt

from cage_fusion import CageFusionConfig, AutoCageFusion
from cage_fusion.data import CageFusionDataModule
from cage_fusion.training import Trainer, TrainingArguments, freeze_phase

## 1. Prepare a new-task dataset

Here we use a toy CSV.  Replace with your own file.

In [ ]:
toy_data = [
    {"SMILES": "CC(=O)Oc1ccccc1C(=O)O",                          "soluble": 1},
    {"SMILES": "c1ccc2ccccc2c1",                                   "soluble": 0},
    {"SMILES": "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",                    "soluble": 1},
    {"SMILES": "CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C",              "soluble": 0},
    {"SMILES": "O=C(O)c1ccccc1O",                                   "soluble": 1},
    {"SMILES": "C1CCCCC1",                                          "soluble": 0},
    {"SMILES": "CCO",                                               "soluble": 1},
    {"SMILES": "CC(C)Cc1ccc(cc1)C(C)C(=O)O",                       "soluble": 1},
    {"SMILES": "CN(C)c1ccc(cc1)C(=C2C=CC(=[N+](C)C)C=C2)c3ccccc3", "soluble": 0},
    {"SMILES": "O=C1c2ccccc2C(=O)c3ccccc13",                        "soluble": 0},
    {"SMILES": "OC(=O)c1ccc(O)cc1",                                 "soluble": 1},
    {"SMILES": "CC(=O)N",                                           "soluble": 1},
]

df = pd.DataFrame(toy_data)
df.to_csv("/tmp/solubility.csv", index=False)
print(f"{len(df)} compounds, label distribution:\n{df['soluble'].value_counts().to_dict()}")

In [ ]:
dm = CageFusionDataModule.from_csv(
    csv_path="/tmp/solubility.csv",
    label_cols=["soluble"],
    model_checkpoint="DeepChem/ChemBERTa-77M-MTR",
    val_split=0.15,
    test_split=0.10,
    cache_dir="/tmp/solubility_features",
    batch_size=8,
)
print("Labels:", dm.label_names)

## 2. Define a new-task config

The task head must match the new task — `num_labels` and `label_names`
will differ from the source checkpoint.

In [ ]:
new_config = CageFusionConfig(
    num_labels=len(dm.label_names),
    model_task="classification",
    label_names=dm.label_names,
    attn_mode="cross",
    hidden_size=128,
)
print(new_config)

## 3. Load encoder weights, reset head

`load_backbone_only=True` sets `strict=False` internally so that the
old task-head weights are simply ignored while the encoder weights
are fully restored.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_ft = AutoCageFusion.from_pretrained(
    "sidxz/cage-fusion-nuisance",    # Hub repo — downloads on first call
    config=new_config,               # new task config with 1 label
    load_backbone_only=True,         # encoder weights loaded; head is fresh
).to(device)

print("Model loaded.  Head is randomly initialised.")

## 4. Option A — one-liner staged fine-tuning

`staged_finetune` runs the canonical 4-phase protocol automatically:

| Phase | Frozen | Trainable |
|---|---|---|
| 0. Warmup | none | all |
| 1. Encoder | aux + fusion | encoder + attention + head |
| 2. Aux warmup | encoder + attn | aux + fusion + head |
| 3. Full | none | all |

In [ ]:
args_ft = TrainingArguments(
    output_dir="/tmp/ft_staged",
    checkpoints_dir="/tmp/ft_staged",
    num_epochs=40,
    batch_size=8,
    learning_rate=1e-3,
    seed=42,
)

trainer_ft = Trainer(
    model=model_ft,
    args=args_ft,
    train_loader=dm.train_loader,
    val_loader=dm.val_loader,
    device=device,
)

trainer_ft.staged_finetune(
    num_epochs_warmup=5,
    num_epochs_phase1=10,
    num_epochs_aux_warmup=5,
    num_epochs_phase2=20,
)

## 4. Option B — manual freeze / unfreeze

Use `freeze_phase` + `rebuild_optimizer` for full control.

In [ ]:
# Reload the backbone (same as above)
model_manual = AutoCageFusion.from_pretrained(
    "sidxz/cage-fusion-nuisance",
    config=new_config,
    load_backbone_only=True,
).to(device)

args_manual = TrainingArguments(
    output_dir="/tmp/ft_manual",
    checkpoints_dir="/tmp/ft_manual",
    num_epochs=40,
    batch_size=8,
    learning_rate=1e-3,
)

trainer_m = Trainer(
    model=model_manual,
    args=args_manual,
    train_loader=dm.train_loader,
    val_loader=dm.val_loader,
    device=device,
)

# Phase 1: freeze aux + fusion, train encoder + head
trainer_m.freeze_phase("freeze_aux_and_fusion")
trainer_m.rebuild_optimizer(lr=1e-3)
for epoch in range(15):
    trainer_m.train_epoch(epoch)

# Phase 2: full unfreeze
trainer_m.freeze_phase("unfreeze_all")
trainer_m.rebuild_optimizer(lr=3e-4)
for epoch in range(15, 30):
    trainer_m.train_epoch(epoch)

print("Manual fine-tuning complete.")

## 5. Compare: fine-tuned vs. trained from scratch

In [ ]:
# Train from scratch for comparison
model_scratch = AutoCageFusion.from_config(new_config).to(device)

args_scratch = TrainingArguments(
    output_dir="/tmp/scratch",
    checkpoints_dir="/tmp/scratch",
    num_epochs=40,
    batch_size=8,
    learning_rate=3e-4,
)

trainer_scratch = Trainer(
    model=model_scratch,
    args=args_scratch,
    train_loader=dm.train_loader,
    val_loader=dm.val_loader,
    device=device,
)

history_scratch = trainer_scratch.train()

# Load the fine-tuned history (saved in checkpoints_dir)
import json, os
# If history was returned from staged_finetune, use it directly.
# Here we compare val AUC trajectories.
print("From scratch   — best val AUC:", max(history_scratch["val_auc"]))

## 6. Save fine-tuned checkpoint

In [ ]:
SAVE_DIR = "/tmp/ft_solubility"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save scaler from the data module
dm.save_scaler(SAVE_DIR)
# Save config
new_config.save_pretrained(SAVE_DIR)

# The Trainer already saved best_model.pt — just check:
print("Checkpoint files:")
for f in sorted(os.listdir(SAVE_DIR)):
    print(" ", f)

In [ ]:
from cage_fusion import CageFusionPipeline

pipe = CageFusionPipeline.from_pretrained(SAVE_DIR)
print(pipe("CCO"))   # ethanol — should be soluble